# Multi-ξ PARF at Full V_φ Capacity (H=128)

## Motivation

The first multi-ξ PARF pilot (H=16, grad-accum=2) reached **15.44 PPL** —
dramatically better than single-ξ PARF (28 PPL) but still above the
standalone multi-ξ SPLM baseline (14.69 PPL).  The ~0.75 PPL overhead
is likely due to V_φ running at crippled capacity (H=16 instead of 128)
because of the old O(B·T²·H) memory scaling.

This notebook re-runs multi-ξ PARF with two memory fixes that eliminate
the V_φ memory bottleneck:

| Fix | Flag | Effect |
|-----|------|--------|
| Level-2 per-layer checkpointing | `--use-layer-checkpoint` | Peak V_φ memory: O(L) → O(1) layers |
| Stage-1.5b gathered V_φ | `--use-gathered-v-phi` | Intermediates: O(T²) → O(T·k), 128× at k=4 |

Combined, these reduce peak V_φ activation memory from ~8 GB to **~8 MB**,
enabling H=128 (full V_φ capacity), no grad-accum, single-pass B=16 on A100.

## Key question

**Do pair-exchange forces improve PPL below the 14.69 multi-ξ SPLM baseline
when V_φ is given adequate capacity?**

## Baselines

| Model | PPL | Steps | V_φ H |
|-------|-----|-------|-------|
| Attention baseline | 7.81 | 8000 | — |
| Multi-ξ SPLM (no PARF) | 14.69 | 4000 | — |
| Multi-ξ PARF (H=16, old) | 15.44 | 4000 | 16 |
| PARF + single ξ | ~28 | 8000 | 16 |

## Arms

| # | Arm | K | α-init | V_φ | k | H | Description |
|---|-----|---|--------|-----|---|---|-------------|
| 1 | `comp_K4_best_alpha` | 4 | [0.25,0.50,0.75,0.95] | competitive | 8 | 128 | Direct re-run of H=16 winner at H=128 |
| 2 | `comp_K4_k4` | 4 | [0.25,0.50,0.75,0.95] | competitive | 4 | 128 | Lower k (faster gathered eval) |
| 3 | `comp_K4_log_spaced` | 4 | log_spaced | competitive | 8 | 128 | Auto-spaced α (no hand-picking) |
| 4 | `comp_K2` | 2 | [0.50,0.95] | competitive | 8 | 128 | Fewer channels (cheaper V_θ) |
| 5 | `comp_K8` | 8 | log_spaced | competitive | 8 | 128 | More channels (higher resolution) |
| 6 | `struct_K4` | 4 | [0.25,0.50,0.75,0.95] | structural | 8 | 128 | Non-competitive V_φ control |

## Hardware

- **A100 40GB / H100 80GB**: ~4-6 h per arm at scaleup (8000 steps);
  faster per-step than old H=16 runs because gathered V_φ is 128× less compute
- No grad-accum needed
- TF32 disabled for autograd.grad stability

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_parf_multixi_h128')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_parf_multixi_h128'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_DIR = REPO_PARENT / 'semsimula-paper'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
PARF_DIR    = REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for PARF autograd.grad stability')
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

All arms use the memory-optimized PARF:
- `--use-layer-checkpoint` (Level-2: O(L) → O(1) layer memory)
- `--use-gathered-v-phi`  (Stage-1.5b: O(T²) → O(T·k) V_φ memory)
- `--v-phi-phi-hidden 128 --v-phi-theta-hidden 128` (full V_φ capacity)
- No `--grad-accum` needed

In [ ]:
MODE          = 'scaleup'     # 8000 steps for fully converged results
FIXED_GAMMA   = 0.30
SEED          = 0
MAX_TRAIN_TOK = 5_000_000
V_PHI_H       = 128           # full V_phi capacity (was 16 in old runs)

ARM_DEFS = {
    'comp_K4_best_alpha': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_inits': '0.25,0.50,0.75,0.95',
        'xi_alpha_init_mode': 'explicit',
        'desc': 'K=4 competitive k=8, best α (H=128 re-run of H=16 winner)',
    },
    'comp_K4_k4': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 4,
        'xi_channels': 4,
        'xi_alpha_inits': '0.25,0.50,0.75,0.95',
        'xi_alpha_init_mode': 'explicit',
        'desc': 'K=4 competitive k=4 (faster gathered, fewer pair forces)',
    },
    'comp_K4_log_spaced': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_inits': None,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'K=4 competitive k=8, log-spaced α (auto, no hand-picking)',
    },
    'comp_K2': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 2,
        'xi_alpha_inits': '0.50,0.95',
        'xi_alpha_init_mode': 'explicit',
        'desc': 'K=2 competitive k=8 (cheaper V_θ, 2-channel summary)',
    },
    'comp_K8': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'xi_channels': 8,
        'xi_alpha_inits': None,
        'xi_alpha_init_mode': 'log_spaced',
        'desc': 'K=8 competitive k=8, log-spaced α (high-resolution ξ)',
    },
    'struct_K4': {
        'v_phi_kind': 'structural',
        'top_k': 8,
        'xi_channels': 4,
        'xi_alpha_inits': '0.25,0.50,0.75,0.95',
        'xi_alpha_init_mode': 'explicit',
        'desc': 'K=4 structural (non-competitive) k=8, best α',
    },
}

ALL_ARMS = list(ARM_DEFS.keys())

print(f'Arms defined : {len(ALL_ARMS)}')
for name in ALL_ARMS:
    print(f'  {name}')

In [ ]:
# ── SELECT ARMS TO RUN ───────────────────────────────────────────────────────
# Edit ARMS_TO_RUN to target any subset of the arms defined above.
# Run this cell after Cell 7 every time you change the selection.
#
# Examples:
#   ARMS_TO_RUN = ALL_ARMS                          # full sweep (default)
#   ARMS_TO_RUN = ['comp_K8']                        # single arm (re-run / debug)
#   ARMS_TO_RUN = ['comp_K4_best_alpha', 'comp_K4_log_spaced']  # two arms
#   ARMS_TO_RUN = [n for n in ALL_ARMS if 'K4' in n] # filter by pattern
# ─────────────────────────────────────────────────────────────────────────────

# Initialise results dict if this is a fresh run
if "results" not in dir():
    results = {}

ARMS_TO_RUN = ALL_ARMS          # ← change this to run a subset

# Validate
invalid = [n for n in ARMS_TO_RUN if n not in ARM_DEFS]
if invalid:
    raise ValueError(f'Unknown arm names: {invalid}\nValid: {ALL_ARMS}')

# Summary
print(f'Memory mode  : Level-2 checkpoint + Stage-1.5b gathered V_φ')
print(f'V_φ capacity : H={V_PHI_H}')
_n_steps = {"scaleup": 8000, "pilot": 4000, "smoke": 300}.get(MODE, "???")
print(f'Schedule     : {MODE}  ({_n_steps} steps)')
print(f'')
print(f'Selected {len(ARMS_TO_RUN)} / {len(ALL_ARMS)} arms:')
for name in ARMS_TO_RUN:
    d = ARM_DEFS[name]
    alpha_str = d['xi_alpha_inits'] or 'log_spaced'
    print(f'  {name:28s}  K={d["xi_channels"]}  k={d["top_k"]}  '
          f'V_φ={d["v_phi_kind"]}  α={alpha_str}')
    print(f'  {"":28s}  {d["desc"]}')



## 4. Precompute logfreq surprisal (if needed)

In [ ]:
LOGFREQ_PATH = SCRIPTS_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'

if not LOGFREQ_PATH.exists():
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / 'compute_unigram_frequencies_tinystories.py')],
        cwd=str(SCRIPTS_DIR), check=True,
    )
    assert LOGFREQ_PATH.exists()
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')

## 5. Train multi-ξ PARF arms (H=128, gathered V_φ)

Each arm trains a `MultiXiPARFLM` with:
- Full V_φ capacity (H=128)
- Level-2 per-layer checkpointing
- Stage-1.5b gathered V_φ evaluation
- No gradient accumulation

Completed arms are skipped on re-run.

In [ ]:
import re
from IPython.display import clear_output

# ── Log-line parsers (match [multixi-parf] prefixed stdout lines) ─────────────
_TRAIN_RE = re.compile(
    r'step\s+(\d+)/(\d+)\s+train\s+([\d.]+)\s+lr\s+([\d.eE+-]+)\s+'
    r'grad\s+([\d.]+)\s+gamma=([\d.]+)\s+tau=([\d.]+)\s+'
    r'[^\[]*\[([^\]]+)\]\s+elapsed\s+([\d.]+)s'
)
_EVAL_RE = re.compile(
    r'>>>\s+eval\s+@\s+(\d+):\s+val\s+([\d.]+)\s+ppl\s+([\d.]+)'
)

def _parse_train(line):
    m = _TRAIN_RE.search(line)
    if not m:
        return None
    return {
        'step': int(m.group(1)), 'total': int(m.group(2)),
        'train_loss': float(m.group(3)), 'lr': float(m.group(4)),
        'grad': float(m.group(5)), 'gamma': float(m.group(6)),
        'tau': float(m.group(7)), 'alphas': m.group(8),
        'elapsed_s': float(m.group(9)),
    }

def _parse_eval(line):
    m = _EVAL_RE.search(line)
    if not m:
        return None
    return {'step': int(m.group(1)), 'val_loss': float(m.group(2)),
            'val_ppl': float(m.group(3))}

def _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines, done=False):
    clear_output(wait=True)
    import sys as _sys; _sys.stdout.flush()
    status = '✓ DONE' if done else '⚙ RUNNING'
    W = 65
    print('═' * W)
    print(f'  {status}  {arm_name}')
    print(f'  {arm_def["desc"]}')
    print('═' * W)

    if tr:
        step, total = tr['step'], tr['total']
        elapsed = time.time() - t_start
        train_elapsed = (time.time() - t_train_start) if t_train_start else elapsed
        pct = 100 * step / total
        eta_s = train_elapsed * (total - step) / step if step > 0 else 0
        eta_str = 'Done!' if done else f'ETA {eta_s / 60:.1f} min'
        bar_w = 52
        filled = int(bar_w * pct / 100)
        bar = '█' * filled + '░' * (bar_w - filled)
        print(f'  [{bar}]')
        print(f'  Step : {step:5d} / {total}  ({pct:.1f}%)'
              f'   Elapsed: {elapsed / 60:.1f} min   {eta_str}')
        print()
        print(f'  train loss : {tr["train_loss"]:.4f}'
              f'   lr : {tr["lr"]:.2e}'
              f'   grad norm : {tr["grad"]:.3f}')
        print(f'  γ          : {tr["gamma"]:.4f}'
              f'   τ (Gumbel): {tr["tau"]:.4f}')
        print(f'  α channels : [{tr["alphas"]}]')

    if ev:
        print()
        print(f'  Last val PPL : {ev["val_ppl"]:.3f}'
              f'   val loss : {ev["val_loss"]:.4f}'
              f'   (@ step {ev["step"]})')

    noise = [l for l in recent_lines
             if l.strip()
             and '[multixi-parf] step' not in l
             and not l.startswith('{')]
    if noise:
        print()
        print('  Recent output:')
        for l in noise[-6:]:
            print(f'    {l}')

def _run_arm_streaming(arm_name, arm_def, cmd):
    """Launch trainer, stream stdout, show live progress.  Returns (returncode, all_lines)."""
    total_steps = {'scaleup': 8000, 'pilot': 4000, 'smoke': 300}.get(MODE, 8000)
    all_lines, recent_lines = [], []
    tr, ev = None, None
    t_start = time.time()
    t_train_start = None  # set on first training step line

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        cwd=str(SCRIPTS_DIR),
        env={**__import__("os").environ, "PYTHONUNBUFFERED": "1"},
    )
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            all_lines.append(line)
            recent_lines = all_lines[-40:]        # rolling window for display

            new_tr = _parse_train(line)
            new_ev = _parse_eval(line)
            if new_tr:
                tr = new_tr
                if t_train_start is None:
                    t_train_start = time.time()  # first step: ignore tokenisation time
            if new_ev:
                ev = new_ev
            if new_tr or new_ev:
                _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines)
            elif not all_lines or tr is None:
                # Before first train line: print setup lines immediately
                if line.strip() and not line.startswith('{'):
                    print(line, flush=True)
    except Exception as exc:
        print(f'\n  STREAM ERROR: {exc}')
    finally:
        proc.wait()

    _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines, done=True)
    return proc.returncode, all_lines


# ── Training loop ─────────────────────────────────────────────────────────────
TRAINER = str(SCRIPTS_DIR / 'train_parf_multixi_scaleup.py')
results = {}

for arm_name in ARMS_TO_RUN:
    arm_def = ARM_DEFS[arm_name]
    arm_results_dir = DRIVE_RESULTS / arm_name
    arm_results_dir.mkdir(parents=True, exist_ok=True)

    # Skip if summary file already exists (completed on a previous run)
    summary_glob = list(arm_results_dir.glob('*_summary.md'))
    if summary_glob:
        print(f'\n⏭  {arm_name}: SKIP (already complete)')
        with open(summary_glob[0]) as f:
            for line in f:
                if 'Final' in line:
                    print(f'   {line.strip()}')
        results[arm_name] = {'status': 'skipped'}
        continue

    cmd = [
        sys.executable, "-u", TRAINER,
        '--mode', MODE,
        '--seed', str(SEED),
        '--fixed-gamma', str(FIXED_GAMMA),
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--results-dir', str(arm_results_dir),
        '--tag-suffix', arm_name,
        '--logfreq-path', str(LOGFREQ_PATH),
        '--device', DEVICE,
        '--v-phi-kind', arm_def['v_phi_kind'],
        '--top-k', str(arm_def['top_k']),
        '--v-phi-phi-hidden', str(V_PHI_H),
        '--v-phi-theta-hidden', str(V_PHI_H),
        '--use-layer-checkpoint',
        '--use-gathered-v-phi',
        '--ln-before-distance',
        '--per-layer-v-phi-scale',
        '--xi-channels', str(arm_def['xi_channels']),
        '--xi-alpha-init-mode', arm_def['xi_alpha_init_mode'],
    ]
    if arm_def['xi_alpha_inits'] is not None:
        cmd.extend(['--xi-alpha-inits', arm_def['xi_alpha_inits']])

    rc, all_lines = _run_arm_streaming(arm_name, arm_def, cmd)

    if rc != 0:
        print(f'\n  ✗ FAILED: trainer exited with code {rc}')
        print('  Last 30 lines of output:')
        for l in all_lines[-30:]:
            print(f'    {l}')
        results[arm_name] = {'status': 'failed', 'returncode': rc}
        continue

    summary_files = list(arm_results_dir.glob('*_summary.md'))
    if summary_files:
        print('\n  ── Training summary ──')
        with open(summary_files[0]) as f:
            print(f.read())

    results[arm_name] = {'status': 'completed'}

# Final status table
print(f'\n{"═"*65}')
print(f'All selected arms finished  ({len(ARMS_TO_RUN)} selected)')
for name, r in results.items():
    icon = {'completed': '✓', 'skipped': '⏭', 'failed': '✗'}.get(r['status'], '?')
    print(f'  {icon}  {name:30s}  {r["status"]}')


## 6. Results comparison

Compare H=128 multi-ξ PARF arms against:
- Multi-ξ SPLM (no PARF): 14.69 PPL
- Multi-ξ PARF (H=16, old): 15.44 PPL
- Attention baseline: 7.81 PPL

In [ ]:
import matplotlib.pyplot as plt

BASELINES = {
    'Multi-ξ SPLM (no PARF)': 14.69,
    'Multi-ξ PARF H=16 (old)': 15.44,
    'PARF + single ξ (P10i)': 28.00,
    'Attention baseline': 7.81,
}

arm_ppls = {}
arm_alphas = {}
arm_details = {}

for arm_name in ARM_DEFS:
    arm_dir = DRIVE_RESULTS / arm_name
    ckpt_files = list(arm_dir.glob('*_ckpt_latest.pt'))
    if not ckpt_files:
        continue
    ckpt = torch.load(ckpt_files[0], map_location='cpu', weights_only=False)
    arm_ppls[arm_name] = ckpt.get('final_val_ppl')
    arm_alphas[arm_name] = ckpt.get('final_xi_alphas')
    arm_details[arm_name] = {
        'n_params': ckpt.get('n_params'),
        'n_v_phi': ckpt.get('n_v_phi_params'),
        'elapsed_sec': ckpt.get('elapsed_sec'),
    }

if not arm_ppls:
    print('No results found yet.')
else:
    print(f'{"Arm":30s} {"K":>3s} {"k":>3s} {"V_φ params":>10s} '
          f'{"Final α":40s} {"PPL":>8s}')
    print('─' * 100)
    for name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        alpha_str = ', '.join(f'{a:.4f}' for a in arm_alphas[name]) if arm_alphas[name] else '?'
        d = ARM_DEFS[name]
        vphi_p = arm_details[name]['n_v_phi'] or 0
        print(f'{name:30s} {d["xi_channels"]:3d} {d["top_k"]:3d} '
              f'{vphi_p:10,d} [{alpha_str:38s}] {arm_ppls[name]:8.2f}')
    print('─' * 100)
    for bname, bppl in BASELINES.items():
        print(f'{bname:30s} {"":>3s} {"":>3s} {"":>10s} '
              f'{"":40s} {bppl:8.2f}  (baseline)')

    # How much did H=128 help?
    best_arm = min(arm_ppls, key=lambda x: arm_ppls[x])
    best_ppl = arm_ppls[best_arm]
    print(f'\nBest arm: {best_arm} → {best_ppl:.2f} PPL')
    if best_ppl < 14.69:
        print(f'  ✓ Beats multi-ξ SPLM baseline (14.69) by {14.69 - best_ppl:.2f} PPL')
        print(f'  → Pair-exchange forces ADD VALUE on top of multi-ξ!')
    else:
        print(f'  Still above multi-ξ SPLM baseline (14.69) by '
              f'{best_ppl - 14.69:.2f} PPL')
    print(f'  Improvement over H=16 multi-ξ PARF (15.44): '
          f'{15.44 - best_ppl:.2f} PPL')

In [ ]:
if arm_ppls:
    all_names = (list(sorted(arm_ppls, key=lambda x: arm_ppls[x]))
                 + list(BASELINES.keys()))
    all_ppls  = ([arm_ppls[n] for n in sorted(arm_ppls, key=lambda x: arm_ppls[x])]
                 + list(BASELINES.values()))
    colors = (['#2ecc71'] * len(arm_ppls)
              + ['#3498db', '#e67e22', '#e74c3c', '#95a5a6'])

    fig, ax = plt.subplots(figsize=(14, max(6, len(all_names) * 0.6)))
    bars = ax.barh(all_names, all_ppls, color=colors, edgecolor='white')
    for bar, ppl in zip(bars, all_ppls):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{ppl:.2f}', va='center', fontsize=9)

    ax.axvline(x=14.69, color='#3498db', linestyle='--', linewidth=1.5,
               label='Multi-ξ SPLM baseline (14.69)')
    ax.set_xlabel('Val PPL (lower is better)')
    ax.set_title(f'Multi-ξ PARF H=128 (gathered V_φ + Level-2 ckpt) — '
                 f'{MODE} mode')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'multixi_parf_h128_comparison.png', dpi=150)
    plt.show()

    # Loss curves overlay
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    for arm_name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        arm_dir = DRIVE_RESULTS / arm_name
        log_files = list(arm_dir.glob('*_training_log.jsonl'))
        if not log_files:
            continue
        steps, vals = [], []
        with open(log_files[0]) as f:
            for line in f:
                rec = json.loads(line)
                if 'val_ppl' in rec:
                    steps.append(rec['step'])
                    vals.append(rec['val_ppl'])
        if steps:
            ax2.plot(steps, vals, marker='o', markersize=4,
                     label=f'{arm_name} ({arm_ppls[arm_name]:.2f})')

    ax2.axhline(y=14.69, color='#3498db', linestyle='--', linewidth=1.5,
                label='Multi-ξ SPLM (14.69)')
    ax2.axhline(y=15.44, color='#e67e22', linestyle=':', linewidth=1.5,
                label='Multi-ξ PARF H=16 (15.44)')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Val PPL')
    ax2.set_title('Convergence: Multi-ξ PARF H=128 arms')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc='upper right')
    fig2.tight_layout()
    fig2.savefig(DRIVE_RESULTS / 'multixi_parf_h128_convergence.png', dpi=150)
    plt.show()

## 7. Save consolidated report

In [ ]:
report = {
    'experiment': 'parf_multixi_h128_gathered',
    'memory_mode': 'Level-2 checkpoint + Stage-1.5b gathered V_phi',
    'v_phi_hidden': V_PHI_H,
    'config': {
        'mode': MODE,
        'fixed_gamma': FIXED_GAMMA,
        'seed': SEED,
        'grad_accum': 1,
        'use_layer_checkpoint': True,
        'use_gathered_v_phi': True,
    },
    'arms': {},
    'baselines': BASELINES,
}

for name in arm_ppls:
    d = ARM_DEFS[name]
    report['arms'][name] = {
        'v_phi_kind': d['v_phi_kind'],
        'top_k': d['top_k'],
        'xi_channels': d['xi_channels'],
        'xi_alpha_init_mode': d['xi_alpha_init_mode'],
        'xi_alpha_inits': d['xi_alpha_inits'],
        'final_ppl': arm_ppls[name],
        'final_alphas': arm_alphas.get(name),
        'n_params': arm_details.get(name, {}).get('n_params'),
        'n_v_phi_params': arm_details.get(name, {}).get('n_v_phi'),
        'elapsed_sec': arm_details.get(name, {}).get('elapsed_sec'),
    }

report_path = DRIVE_RESULTS / 'parf_multixi_h128_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved: {report_path}')
if IN_COLAB:
    print('Results are persisted on Google Drive at:')
    print(f'  {DRIVE_ROOT}')